In [1]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
import time
import pandas as pd


# Driver setup
driver = webdriver.Chrome()

# Movie genres
genres = ["Adventure", "Animation", "Family", "Fantasy", "Mystery"]

# Movies collection
Movielist_df = pd.DataFrame()

for genre in genres:
    url = f"https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31&genres={genre}"
    driver.get(url)
    time.sleep(5)

    def click_load_more():
        try:
            load_more_button = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/div[2]/div/span/button/span/span')
            ActionChains(driver).move_to_element(load_more_button).perform()
            load_more_button.click()
            time.sleep(5)
            return True
        except Exception as e:
            print("No more content to load or error:", e)
            return False

    while click_load_more():
        print("Clicked 'Load More' button")

    print("Finished loading all movies for", genre)

    titles = []
    ratings = []
    votings = []
    durations = []

    movie_items = driver.find_elements(By.XPATH, '//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li')

    for movie_item in movie_items:
        try:
            title = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/div[1]/a/h3').text
            rating = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/span/div/span/span[1]').text
            voting = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/span/div/span/span[2]').text
            duration = movie_item.find_element(By.XPATH, './div/div/div/div[1]/div[2]/div[2]/span[2]').text

            titles.append(title)
            ratings.append(rating)
            votings.append(voting)
            durations.append(duration)

        except Exception as e:
            print(f"Error extracting data for a movie: {e}")
            continue

    df = pd.DataFrame({
        'Title': titles,
        'Rating': ratings,
        'Votes': votings,
        'Duration': durations,
        'Genre': genre
    })

    # Clean up Title and Votes fields
    df['Title'] = df['Title'].str.replace(r'^\d+\.\s*', '', regex=True)
    df['Votes'] = df['Votes'].str.replace(r'[\(\)]', '', regex=True)

    # Save individual genre CSV
    df.to_csv(f"{genre}_2024_movies.csv", index=False)

    # Add to the final DataFrame for all genres
    Movielist_df = pd.concat([Movielist_df, df], ignore_index=True)

# Save combined CSV for all genres
Movielist_df.to_csv("all_genres_2024_movies.csv", index=False)
print("\n All genres saved to all_genres_2024_movies.csv")

driver.quit()


Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
Clicked 'Load More' button
No more content to load or error: Message: no such element: Unable to locate element: {"method":"xpath","selector":"//*[@id="__next"]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/div[2]/div/span/button/span/span"}
  (Session info: chrome=135.0.7049.115); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#no-such-element-exception
Stacktrace:
	GetHandleVerifier [0x00007FF66CFEEFA5+77893]
	GetHandleVerifier [0x00007FF66CFEF000+77984]
	(No symbol) [0x00007FF66CDB91BA]
	(No symbol) [0x00007FF66CE0F16D]
	(No symbol) [0x00007FF66CE0F41C]
	(No symbol) [0x00007FF66CE

In [11]:
import re

# Function to convert duration to total minutes as int
def convert_duration_to_minutes(duration):
    # Check if duration is already a number
    if isinstance(duration, (int, float)):
        return int(duration)
    
    # If not, process it as a string
    duration = str(duration).lower().strip()
    hours = minutes = 0
    hr_match = re.search(r'(\d+)\s*h', duration)
    min_match = re.search(r'(\d+)\s*m', duration)
    
    if hr_match:
        hours = int(hr_match.group(1))
    if min_match:
        minutes = int(min_match.group(1))
    
    return hours * 60 + minutes

# Function to convert vote strings like "53K" to integer
def convert_votes_to_int(votes):
    # Check if votes is already a number
    if isinstance(votes, (int, float)):
        return int(votes)
    
    # If not, process it as a string
    votes = str(votes).strip().upper()
    
    if 'K' in votes:
        return int(float(votes.replace('K', '')) * 1000)
    elif 'M' in votes:
        return int(float(votes.replace('M', '')) * 1000000)
    
    return int(votes)

# Apply the conversion functions to your DataFrame
Movielist_df['Duration'] = Movielist_df['Duration'].apply(convert_duration_to_minutes)
Movielist_df['Votes'] = Movielist_df['Votes'].apply(convert_votes_to_int)

# Show the first few rows
Movielist_df.head()


,Title,Rating,Votes,Duration,Genre
0,Mufasa: The Lion King,6.6,64000,118,Adventure
1,Twisters,6.5,169000,122,Adventure
2,Gladiator II,6.5,230000,148,Adventure
3,Moana 2,6.6,102000,100,Adventure
4,Flow,7.9,75000,85,Adventure


In [12]:
pip install sqlalchemy pymysql


  Using cached PyMySQL-1.1.1-py3-none-any.whl.metadata (4.4 kB)
Using cached PyMySQL-1.1.1-py3-none-any.whl (44 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from sqlalchemy import create_engine

# Replace these with your actual credentials
username = 'root'             # or your username
password = 9618172007         # your MySQL password
host = 'localhost'            # or your host IP
database = 'imdb_movies'      # make sure this database exists

# Create SQLAlchemy engine
engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}/{database}')

# Store DataFrame in SQL
Movielist_df.to_sql(name='movies_2024', con=engine, if_exists='replace', index=False)

print(" Data successfully saved into SQL database!")


 Data successfully saved into SQL database!
